In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/nba_final.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)


df = df.dropna(subset=['roll_pts_scored', 'roll_win_rate', 'team_elo'])

print("Shape after dropping NaN:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())

Shape after dropping NaN: (31254, 41)
Date range: 2012-10-30 00:00:00 to 2025-04-13 00:00:00


In [2]:

FEATURES = [
    'roll_pts_scored',   
    'roll_plus_minus',    
    'roll_win_rate',     
    'roll_fg_pct',        
    'days_rest',          
    'streak',             
    'team_elo',           
    'opp_elo',            
    'elo_diff',           
    'h2h_win_rate',       
    'is_home',            
]

X = df[FEATURES]
y = df['result']


split_idx = int(len(df) * 0.80)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f"Training on {len(X_train)} games")
print(f"Testing on  {len(X_test)} games")
print(f"\nTrain win rate: {y_train.mean():.3f}")
print(f"Test win rate:  {y_test.mean():.3f}")


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Training on 25003 games
Testing on  6251 games

Train win rate: 0.501
Test win rate:  0.500


In [3]:
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("✓ Done!")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_prob_lr):.3f}")

Training Logistic Regression...
✓ Done!
  Accuracy : 0.671
  ROC-AUC  : 0.737


In [4]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest (may take 1-2 minutes)...")
rf = RandomForestClassifier(
    n_estimators=200,     
    max_depth=8,          
    min_samples_leaf=20,  
    random_state=42,
    n_jobs=-1             
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("✓ Done!")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.3f}")

Training Random Forest (may take 1-2 minutes)...
✓ Done!
  Accuracy : 0.670
  ROC-AUC  : 0.734


In [5]:
import xgboost as xgb

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("✓ Done!")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_xgb):.3f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_prob_xgb):.3f}")

Training XGBoost...
✓ Done!
  Accuracy : 0.673
  ROC-AUC  : 0.732


In [6]:
results = pd.DataFrame({
    'Model':    ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_lr),  3),
        round(accuracy_score(y_test, y_pred_rf),  3),
        round(accuracy_score(y_test, y_pred_xgb), 3),
    ],
    'ROC_AUC': [
        round(roc_auc_score(y_test, y_prob_lr),  3),
        round(roc_auc_score(y_test, y_prob_rf),  3),
        round(roc_auc_score(y_test, y_prob_xgb), 3),
    ]
})
results = results.sort_values('ROC_AUC', ascending=False)
print("=" * 45)
print("         MODEL COMPARISON RESULTS")
print("=" * 45)
print(results.to_string(index=False))
print("=" * 45)
print(f"\nBaseline (always pick home team): ~0.580")
print(f"Your best model beats baseline by: {results['Accuracy'].max() - 0.58:.3f}")

         MODEL COMPARISON RESULTS
              Model  Accuracy  ROC_AUC
Logistic Regression     0.671    0.737
      Random Forest     0.670    0.734
            XGBoost     0.673    0.732

Baseline (always pick home team): ~0.580
Your best model beats baseline by: 0.093


In [7]:
import os
os.makedirs('../src', exist_ok=True)


joblib.dump(lr,        '../src/logistic_regression.pkl')
joblib.dump(rf,        '../src/random_forest.pkl')
joblib.dump(xgb_model, '../src/xgboost_model.pkl')
joblib.dump(scaler,    '../src/scaler.pkl')


import json
with open('../src/features.json', 'w') as f:
    json.dump(FEATURES, f)

print("All models saved to src/ folder!")
print("Files saved:")
for f in os.listdir('../src'):
    print(f"  - {f}")

All models saved to src/ folder!
Files saved:
  - features.json
  - logistic_regression.pkl
  - random_forest.pkl
  - scaler.pkl
  - xgboost_model.pkl
